# Phase 3 — Baselines and Team Strength

## 1. Load dataset

In [ ]:
from pathlib import Path
import pandas as pd
from fpl_predictor.config import HISTORICAL_ML_DIR, HISTORICAL_RAW_DIR, HISTORICAL_SEASONS
from fpl_predictor.eligibility import EligibilityRules, eligibility_mask
from fpl_predictor.feature_sets import FEATURE_SETS
from fpl_predictor.splits import season_holdout_split
from fpl_predictor.team_strength import calculate_team_strength, load_historical_team_matches

dataset = pd.read_csv(HISTORICAL_ML_DIR / 'player_gameweek_dataset.csv', low_memory=False)
results = pd.read_csv(HISTORICAL_ML_DIR / 'baseline_results.csv')
by_position = pd.read_csv(HISTORICAL_ML_DIR / 'baseline_results_by_position.csv')
by_gw = pd.read_csv(HISTORICAL_ML_DIR / 'baseline_results_by_gw.csv')

## 2. Temporal split

In [ ]:
split = season_holdout_split(dataset)
{'train': sorted(split.train.season.unique()), 'validation': sorted(split.validation.season.unique()), 'test': sorted(split.test.season.unique())}

## 3. Baseline overview

In [ ]:
display(results[['split', 'model', 'sample_count']])
display(pd.Series({name: len(features) for name, features in FEATURE_SETS.items()}, name='feature_count'))

## 4. Aggregate model comparison

In [ ]:
display(results.sort_values(['split', 'mae'])[['split', 'model', 'mae', 'rmse', 'spearman', 'mean_actual', 'mean_predicted']])

## 5. Performance by position

In [ ]:
display(by_position.query("split == 'test'").pivot(index='model', columns='position', values='mae').round(3))

## 6. Gameweek-level ranking performance

In [ ]:
display(by_gw.query("split == 'test'").groupby('model')[['spearman', 'mae']].mean().sort_values('mae'))

## 7. Top-K evaluation

In [ ]:
display(results.query("split == 'test'")[["model", "top_10_precision", "top_25_precision", "top_50_precision", "top_10_recall", "top_25_recall", "top_50_recall"]])

## 8. Top-pick regret

In [ ]:
display(results.query("split == 'test'")[["model", "average_top_pick_points", "average_actual_maximum", "average_regret"]].sort_values('average_regret'))

## 9. Team-strength inspection

In [ ]:
matches = load_historical_team_matches(HISTORICAL_RAW_DIR, HISTORICAL_SEASONS)
ratings = calculate_team_strength(matches)
display(ratings.query("season == '2025-26' and gw == 20").sort_values('team_attack_strength_5_rel', ascending=False).head(10))

## 10. Feature missingness

In [ ]:
full_features = [feature for feature in FEATURE_SETS['feature_set_full_linear'] if feature in dataset]
display(dataset[full_features].isna().mean().sort_values(ascending=False).head(20).to_frame('missing_fraction'))

## 11. Ridge benchmark

In [ ]:
display(results.query("model == 'Ridge'")[["split", "mae", "rmse", "spearman", "top_25_precision", "average_regret"]])

## 12. Observations

Compare validation and held-out test behavior before changing the feature set. Position-mean within-position Spearman is undefined because every player in a position receives the same prediction. Team defensive strength is goals conceded, so higher relative values indicate weaker defense. These experiments are benchmarks, not a production expected-points model.